# Model Phát Hiện Nội Dung Nhạy Cảm - XLM-RoBERTa Improved

Notebook này giữ nguyên base model **`xlm-roberta-base`**, nhưng cải thiện pipeline theo hướng production hơn:

- Chuẩn hóa Unicode NFC cho tiếng Việt
- Xử lý text cơ bản và teencode/slang mapping
- Multi-label classification: `clean`, `hate`, `sexual`, `spam`, `toxic`
- Xử lý mất cân bằng nhãn bằng **Weighted Loss** hoặc **Focal Loss**
- Differential Learning Rate: LR thấp cho base model, LR cao hơn cho classifier
- Early stopping
- Tìm threshold tối ưu riêng cho từng label dựa trên validation F1
- Inference độc lập, không phụ thuộc biến training như `test_texts`
- Logic hậu xử lý: `ALLOW`, `REVIEW`, `BLOCK`
- Lưu model, tokenizer, thresholds, label mapping và zip model
- Optional: export ONNX + quantization INT8

In [ ]:
# Cài thư viện cần thiết
!pip -q install transformers datasets accelerate scikit-learn pandas numpy evaluate iterative-stratification onnx onnxruntime onnxruntime-tools

## 1. Import thư viện và cấu hình chung

In [ ]:
import os
import re
import json
import shutil
import random
import unicodedata
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset

from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)

try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
    HAS_ITERATIVE_STRATIFY = True
except Exception:
    HAS_ITERATIVE_STRATIFY = False

# =====================
# CONFIG
# =====================
MODEL_NAME = 'xlm-roberta-base'
LABELS = ['clean', 'hate', 'sexual', 'spam', 'toxic']
ID2LABEL = {i: label for i, label in enumerate(LABELS)}
LABEL2ID = {label: i for i, label in enumerate(LABELS)}

TEXT_COLUMN = 'text'  # đổi nếu file CSV của bạn dùng tên cột khác, ví dụ: 'content', 'caption'
DATA_PATH = '/content/moderation_dataset.csv'  # upload file CSV lên Colab rồi chỉnh path nếu cần

MAX_LENGTH = 128
SEED = 42
OUTPUT_ROOT = Path('/content/moderation_xlm_roberta_output')
MODEL_DIR = OUTPUT_ROOT / 'model'
ARTIFACT_DIR = OUTPUT_ROOT / 'artifacts'

# Training config
NUM_EPOCHS = 5
BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 1
BASE_LR = 2e-5
CLASSIFIER_LR = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

# Loss config: chọn 'weighted_bce' hoặc 'focal'
LOSS_TYPE = 'weighted_bce'
FOCAL_GAMMA = 2.0

# Decision config
REVIEW_LOW = 0.40
REVIEW_HIGH = 0.65
BLOCK_THRESHOLD = 0.65

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('Has iterative stratify:', HAS_ITERATIVE_STRATIFY)

## 2. Mount Google Drive (tuỳ chọn)

Nếu bạn muốn đọc dataset hoặc lưu model vào Google Drive, chạy cell này. Nếu không dùng Drive thì có thể bỏ qua.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Không chạy trên Colab hoặc không cần mount Drive:', e)

## 3. Load dataset

Dataset cần có dạng CSV với ít nhất:

- Một cột text, mặc định là `text`
- Các cột label: `clean`, `hate`, `sexual`, `spam`, `toxic`

Mỗi label nên là `0` hoặc `1`.

In [ ]:
# Nếu muốn upload trực tiếp từ máy lên Colab, bỏ comment block dưới đây:
# from google.colab import files
# uploaded = files.upload()
# DATA_PATH = list(uploaded.keys())[0]

if not Path(DATA_PATH).exists():
    raise FileNotFoundError(
        f'Không tìm thấy DATA_PATH={DATA_PATH}. Hãy upload CSV lên Colab hoặc chỉnh DATA_PATH.'
    )

df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
display(df.head())

missing_cols = [c for c in [TEXT_COLUMN] + LABELS if c not in df.columns]
if missing_cols:
    raise ValueError(f'Thiếu các cột sau trong dataset: {missing_cols}')

# Ép label về int 0/1
for col in LABELS:
    df[col] = df[col].fillna(0).astype(int).clip(0, 1)

# Xóa dòng text rỗng
df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna('').astype(str)
df = df[df[TEXT_COLUMN].str.strip().str.len() > 0].reset_index(drop=True)

print('After cleanup:', df.shape)
print('Label distribution:')
display(df[LABELS].sum().to_frame('count'))

## 4. Tiền xử lý tiếng Việt + slang/teencode mapping

Lưu ý: không nên normalize quá mạnh vì moderation cần giữ lại nhiều dấu hiệu bất thường. Cell này chỉ xử lý cơ bản:

- Unicode NFC
- Lowercase
- Chuẩn hóa khoảng trắng
- Mapping một số teencode/slang dễ gặp
- Chuẩn hóa các cách viết lách luật đơn giản

In [ ]:
SLANG_MAP = {
    # Ví dụ cơ bản, bạn nên bổ sung theo dữ liệu thực tế của app
    'n.g.u': 'ngu',
    'n_g_u': 'ngu',
    'n-g-u': 'ngu',
    'nqu': 'ngu',
    'nguu': 'ngu',
    'đm': 'đụ má',
    'dm': 'đụ má',
    'd m': 'đụ má',
    'vcl': 'vãi cả lồn',
    'vl': 'vãi lồn',
    'clm': 'con lợn mẹ',
}

def normalize_unicode(text: str) -> str:
    return unicodedata.normalize('NFC', str(text))

def normalize_repeated_chars(text: str, max_repeat: int = 2) -> str:
    # ví dụ: nguuuuu -> nguu, đẹpppp -> đẹp p? Giữ tối đa 2 ký tự lặp để không phá tiếng Việt quá mạnh
    return re.sub(r'(.){' + str(max_repeat) + r',}', r'' * max_repeat, text)

def apply_slang_mapping(text: str) -> str:
    padded = f' {text} '
    for src, tgt in SLANG_MAP.items():
        padded = re.sub(rf'(?<!\w){re.escape(src)}(?!\w)', tgt, padded, flags=re.IGNORECASE)
    return padded.strip()

def clean_text(text: str) -> str:
    text = normalize_unicode(text)
    text = text.lower()
    text = text.replace('​', ' ')
    text = re.sub(r'http\S+|www\.\S+', ' URL ', text)
    text = re.sub(r'@\w+', ' USER ', text)
    text = apply_slang_mapping(text)
    text = normalize_repeated_chars(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_text'] = df[TEXT_COLUMN].apply(clean_text)

display(df[[TEXT_COLUMN, 'cleaned_text'] + LABELS].head(10))

## 5. Kiểm tra logic label

Với multi-label moderation, `clean` thường nên bằng `1` khi tất cả label nhạy cảm bằng `0`. Nếu dataset của bạn chưa nhất quán, cell dưới sẽ tự sửa.

In [ ]:
BAD_LABELS = ['hate', 'sexual', 'spam', 'toxic']

# Nếu bất kỳ nhãn xấu nào = 1 thì clean phải = 0
bad_any = df[BAD_LABELS].sum(axis=1) > 0
df.loc[bad_any, 'clean'] = 0

# Nếu không có nhãn xấu nào thì clean = 1
no_bad = df[BAD_LABELS].sum(axis=1) == 0
df.loc[no_bad, 'clean'] = 1

print('Label distribution after label consistency fix:')
display(df[LABELS].sum().to_frame('count'))

## 6. Train / Validation / Test split

Ưu tiên dùng multilabel stratified split để giữ phân phối nhãn ổn định giữa train/val/test.

In [ ]:
def multilabel_split(dataframe: pd.DataFrame, label_cols: List[str], seed: int = 42):
    X = dataframe.index.values
    y = dataframe[label_cols].values

    if HAS_ITERATIVE_STRATIFY:
        splitter1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
        train_idx, temp_idx = next(splitter1.split(X, y))

        temp_df = dataframe.iloc[temp_idx].reset_index(drop=True)
        X_temp = temp_df.index.values
        y_temp = temp_df[label_cols].values
        splitter2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=seed)
        val_idx_local, test_idx_local = next(splitter2.split(X_temp, y_temp))

        train_df = dataframe.iloc[train_idx].reset_index(drop=True)
        val_df = temp_df.iloc[val_idx_local].reset_index(drop=True)
        test_df = temp_df.iloc[test_idx_local].reset_index(drop=True)
    else:
        train_df, temp_df = train_test_split(dataframe, test_size=0.2, random_state=seed)
        val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=seed)
        train_df = train_df.reset_index(drop=True)
        val_df = val_df.reset_index(drop=True)
        test_df = test_df.reset_index(drop=True)

    return train_df, val_df, test_df

train_df, val_df, test_df = multilabel_split(df, LABELS, SEED)

print('Train:', train_df.shape)
print('Val:', val_df.shape)
print('Test:', test_df.shape)

print('
Train label distribution:')
display(train_df[LABELS].sum().to_frame('train_count'))
print('
Val label distribution:')
display(val_df[LABELS].sum().to_frame('val_count'))
print('
Test label distribution:')
display(test_df[LABELS].sum().to_frame('test_count'))

## 7. Dataset class + tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ModerationDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, tokenizer, text_col: str, label_cols: List[str], max_length: int):
        self.texts = dataframe[text_col].astype(str).tolist()
        self.labels = dataframe[label_cols].astype(float).values
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
        )
        encoding['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return encoding

train_dataset = ModerationDataset(train_df, tokenizer, 'cleaned_text', LABELS, MAX_LENGTH)
val_dataset = ModerationDataset(val_df, tokenizer, 'cleaned_text', LABELS, MAX_LENGTH)
test_dataset = ModerationDataset(test_df, tokenizer, 'cleaned_text', LABELS, MAX_LENGTH)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 8. Tính `pos_weight` cho Weighted Loss

`pos_weight` giúp mô hình chú ý hơn tới các nhãn hiếm như `hate`, `sexual`, `toxic`.

In [ ]:
train_labels = train_df[LABELS].values.astype(np.float32)
pos_counts = train_labels.sum(axis=0)
neg_counts = len(train_labels) - pos_counts
pos_weight = neg_counts / np.clip(pos_counts, 1, None)
pos_weight = np.clip(pos_weight, 1.0, 20.0)  # tránh weight quá lớn gây mất ổn định
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float)

for label, weight, pos, neg in zip(LABELS, pos_weight, pos_counts, neg_counts):
    print(f'{label:8s} pos={int(pos):5d} neg={int(neg):5d} pos_weight={weight:.3f}')

## 9. Custom Trainer: Weighted BCE / Focal Loss + Differential Learning Rate

In [ ]:
class FocalLossWithLogits(nn.Module):
    def __init__(self, pos_weight=None, gamma: float = 2.0, reduction: str = 'mean'):
        super().__init__()
        self.pos_weight = pos_weight
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits,
            targets,
            pos_weight=self.pos_weight.to(logits.device) if self.pos_weight is not None else None,
            reduction='none',
        )
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        focal_factor = (1 - p_t) ** self.gamma
        loss = focal_factor * bce
        if self.reduction == 'mean':
            return loss.mean()
        if self.reduction == 'sum':
            return loss.sum()
        return loss

class ModerationTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, loss_type='weighted_bce', focal_gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight
        self.loss_type = loss_type
        self.focal_gamma = focal_gamma

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits

        if self.loss_type == 'focal':
            loss_fn = FocalLossWithLogits(pos_weight=self.pos_weight, gamma=self.focal_gamma)
        else:
            loss_fn = nn.BCEWithLogitsLoss(
                pos_weight=self.pos_weight.to(logits.device) if self.pos_weight is not None else None
            )

        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

    def create_optimizer(self):
        if self.optimizer is None:
            classifier_params = []
            base_params = []
            for name, param in self.model.named_parameters():
                if not param.requires_grad:
                    continue
                if 'classifier' in name or 'score' in name:
                    classifier_params.append(param)
                else:
                    base_params.append(param)

            optimizer_grouped_parameters = [
                {'params': base_params, 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY},
                {'params': classifier_params, 'lr': CLASSIFIER_LR, 'weight_decay': WEIGHT_DECAY},
            ]
            self.optimizer = torch.optim.AdamW(optimizer_grouped_parameters)
        return self.optimizer

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = sigmoid(logits)
    preds = (probs >= 0.5).astype(int)

    return {
        'micro_f1_0_5': f1_score(labels, preds, average='micro', zero_division=0),
        'macro_f1_0_5': f1_score(labels, preds, average='macro', zero_division=0),
        'micro_precision_0_5': precision_score(labels, preds, average='micro', zero_division=0),
        'micro_recall_0_5': recall_score(labels, preds, average='micro', zero_division=0),
    }

## 10. Khởi tạo model và training

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    problem_type='multi_label_classification',
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_ROOT / 'checkpoints'),
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=50,
    learning_rate=BASE_LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type='cosine',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1_0_5',
    greater_is_better=True,
    save_total_limit=2,
    report_to='none',
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

trainer = ModerationTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    pos_weight=pos_weight_tensor,
    loss_type=LOSS_TYPE,
    focal_gamma=FOCAL_GAMMA,
)

trainer.train()

## 11. Tìm threshold tối ưu cho từng label trên validation set

Không dùng chung threshold `0.5` cho tất cả nhãn. Mỗi nhãn có thể cần độ nhạy khác nhau.

In [ ]:
def find_best_thresholds(y_true: np.ndarray, y_prob: np.ndarray, labels: List[str], low=0.1, high=0.9, step=0.01):
    thresholds = {}
    report_rows = []
    candidates = np.arange(low, high + 1e-9, step)

    for i, label in enumerate(labels):
        best_t = 0.5
        best_f1 = -1
        for t in candidates:
            pred = (y_prob[:, i] >= t).astype(int)
            score = f1_score(y_true[:, i], pred, zero_division=0)
            if score > best_f1:
                best_f1 = score
                best_t = float(t)

        thresholds[label] = round(best_t, 2)
        final_pred = (y_prob[:, i] >= best_t).astype(int)
        report_rows.append({
            'label': label,
            'threshold': round(best_t, 2),
            'f1': f1_score(y_true[:, i], final_pred, zero_division=0),
            'precision': precision_score(y_true[:, i], final_pred, zero_division=0),
            'recall': recall_score(y_true[:, i], final_pred, zero_division=0),
            'support': int(y_true[:, i].sum()),
        })
    return thresholds, pd.DataFrame(report_rows)

val_pred = trainer.predict(val_dataset)
val_logits = val_pred.predictions
val_probs = sigmoid(val_logits)
val_true = val_df[LABELS].values.astype(int)

best_thresholds, threshold_report = find_best_thresholds(val_true, val_probs, LABELS)
print('Best thresholds:', best_thresholds)
display(threshold_report)

with open(ARTIFACT_DIR / 'thresholds.json', 'w', encoding='utf-8') as f:
    json.dump(best_thresholds, f, ensure_ascii=False, indent=2)

## 12. Đánh giá trên test set với threshold tối ưu

In [ ]:
def predict_with_thresholds(probs: np.ndarray, thresholds: Dict[str, float], labels: List[str]):
    preds = np.zeros_like(probs, dtype=int)
    for i, label in enumerate(labels):
        preds[:, i] = (probs[:, i] >= thresholds[label]).astype(int)
    return preds

test_pred = trainer.predict(test_dataset)
test_logits = test_pred.predictions
test_probs = sigmoid(test_logits)
test_true = test_df[LABELS].values.astype(int)
test_preds = predict_with_thresholds(test_probs, best_thresholds, LABELS)

print('Micro F1:', f1_score(test_true, test_preds, average='micro', zero_division=0))
print('Macro F1:', f1_score(test_true, test_preds, average='macro', zero_division=0))
print('
Classification report:')
print(classification_report(test_true, test_preds, target_names=LABELS, zero_division=0))

# Lưu test predictions để review lỗi
pred_cols = [f'pred_{l}' for l in LABELS]
prob_cols = [f'prob_{l}' for l in LABELS]
review_df = test_df[[TEXT_COLUMN, 'cleaned_text'] + LABELS].copy()
for i, label in enumerate(LABELS):
    review_df[pred_cols[i]] = test_preds[:, i]
    review_df[prob_cols[i]] = test_probs[:, i]

review_df['is_wrong'] = (test_true != test_preds).any(axis=1)
review_df.to_csv(ARTIFACT_DIR / 'test_predictions_for_error_analysis.csv', index=False, encoding='utf-8-sig')

print('Saved error analysis file:', ARTIFACT_DIR / 'test_predictions_for_error_analysis.csv')
display(review_df.head(20))

## 13. Hard Negative Mining

Cell này lấy các mẫu model đoán sai để bạn review thủ công. Đây là cách cải thiện dataset rất hiệu quả.

In [ ]:
wrong_df = review_df[review_df['is_wrong']].copy()
wrong_df = wrong_df.sort_values(by=prob_cols, ascending=False)
wrong_path = ARTIFACT_DIR / 'hard_cases_to_review.csv'
wrong_df.to_csv(wrong_path, index=False, encoding='utf-8-sig')

print('Số mẫu đoán sai:', len(wrong_df))
print('Saved:', wrong_path)
display(wrong_df.head(30))

## 14. Lưu model, tokenizer, metadata

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))

metadata = {
    'model_name': MODEL_NAME,
    'labels': LABELS,
    'id2label': ID2LABEL,
    'label2id': LABEL2ID,
    'max_length': MAX_LENGTH,
    'loss_type': LOSS_TYPE,
    'base_lr': BASE_LR,
    'classifier_lr': CLASSIFIER_LR,
    'thresholds': best_thresholds,
    'review_low': REVIEW_LOW,
    'review_high': REVIEW_HIGH,
    'block_threshold': BLOCK_THRESHOLD,
}

with open(MODEL_DIR / 'moderation_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

with open(MODEL_DIR / 'slang_map.json', 'w', encoding='utf-8') as f:
    json.dump(SLANG_MAP, f, ensure_ascii=False, indent=2)

print('Saved model to:', MODEL_DIR)
print('Files:', os.listdir(MODEL_DIR))

## 15. Inference độc lập

Phần này có thể chạy riêng sau khi đã lưu model. Không phụ thuộc `test_texts`, `test_labels` hay dataframe training.

In [ ]:
class ModerationPredictor:
    def __init__(self, model_dir: str):
        self.model_dir = Path(model_dir)
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_dir)
        self.model = AutoModelForSequenceClassification.from_pretrained(self.model_dir)
        self.model.eval()
        self.model.to(DEVICE)

        metadata_path = self.model_dir / 'moderation_metadata.json'
        if metadata_path.exists():
            with open(metadata_path, 'r', encoding='utf-8') as f:
                self.metadata = json.load(f)
        else:
            self.metadata = {}

        self.labels = self.metadata.get('labels', LABELS)
        self.thresholds = self.metadata.get('thresholds', {label: 0.5 for label in self.labels})
        self.max_length = self.metadata.get('max_length', MAX_LENGTH)

    def predict_proba(self, texts: List[str]) -> np.ndarray:
        cleaned = [clean_text(t) for t in texts]
        encoded = self.tokenizer(
            cleaned,
            truncation=True,
            padding=True,
            max_length=self.max_length,
            return_tensors='pt',
        ).to(DEVICE)

        with torch.no_grad():
            logits = self.model(**encoded).logits.detach().cpu().numpy()
        return sigmoid(logits)

    def predict(self, texts: List[str]) -> List[Dict]:
        probs = self.predict_proba(texts)
        results = []

        for text, prob in zip(texts, probs):
            scores = {label: float(prob[i]) for i, label in enumerate(self.labels)}
            triggered = [label for label in self.labels if scores[label] >= self.thresholds.get(label, 0.5)]

            # Nếu có label xấu thì bỏ clean
            bad_triggered = [l for l in triggered if l != 'clean']
            if bad_triggered:
                triggered = bad_triggered

            max_bad_score = max([scores[l] for l in self.labels if l != 'clean'], default=0.0)

            if max_bad_score >= BLOCK_THRESHOLD:
                action = 'BLOCK'
            elif max_bad_score >= REVIEW_LOW:
                action = 'REVIEW'
            else:
                action = 'ALLOW'

            results.append({
                'text': text,
                'cleaned_text': clean_text(text),
                'action': action,
                'labels': triggered if triggered else ['clean'],
                'scores': scores,
            })
        return results

predictor = ModerationPredictor(str(MODEL_DIR))

sample_texts = [
    'Bạn hôm nay đẹp quá',
    'Mày ngu thật đấy',
    'Click vào link này để nhận quà miễn phí!!!',
    'Nội dung bình thường không có gì nhạy cảm',
]

results = predictor.predict(sample_texts)
for r in results:
    print(json.dumps(r, ensure_ascii=False, indent=2))

## 16. Zip model để download hoặc deploy

In [ ]:
zip_path = OUTPUT_ROOT / 'moderation_xlm_roberta_model.zip'
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', MODEL_DIR)
print('Zip saved:', zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print('Không chạy trên Colab hoặc không cần download tự động:', e)

## 17. Optional: Export ONNX

Phần này dùng khi bạn muốn deploy tối ưu hơn. Có thể chạy sau khi model đã ổn.

In [ ]:
# Optional ONNX export
# Nếu chưa cần deploy production thì có thể bỏ qua cell này.

RUN_ONNX_EXPORT = False

if RUN_ONNX_EXPORT:
    onnx_dir = OUTPUT_ROOT / 'onnx'
    onnx_dir.mkdir(parents=True, exist_ok=True)
    onnx_path = onnx_dir / 'model.onnx'

    dummy_text = 'xin chào'
    dummy = tokenizer(dummy_text, return_tensors='pt', padding=True, truncation=True, max_length=MAX_LENGTH)
    dummy = {k: v.to(DEVICE) for k, v in dummy.items()}
    model.to(DEVICE)
    model.eval()

    torch.onnx.export(
        model,
        args=(dummy['input_ids'], dummy['attention_mask']),
        f=str(onnx_path),
        input_names=['input_ids', 'attention_mask'],
        output_names=['logits'],
        dynamic_axes={
            'input_ids': {0: 'batch_size', 1: 'sequence'},
            'attention_mask': {0: 'batch_size', 1: 'sequence'},
            'logits': {0: 'batch_size'},
        },
        opset_version=14,
    )
    print('ONNX saved:', onnx_path)

## 18. Optional: Quantization INT8 cho ONNX

Chỉ chạy sau khi đã export ONNX thành công.

In [ ]:
# Optional INT8 quantization
RUN_INT8_QUANTIZATION = False

if RUN_INT8_QUANTIZATION:
    from onnxruntime.quantization import quantize_dynamic, QuantType

    onnx_path = OUTPUT_ROOT / 'onnx' / 'model.onnx'
    int8_path = OUTPUT_ROOT / 'onnx' / 'model_int8.onnx'

    if not onnx_path.exists():
        raise FileNotFoundError('Chưa có model.onnx. Hãy chạy cell export ONNX trước.')

    quantize_dynamic(
        model_input=str(onnx_path),
        model_output=str(int8_path),
        weight_type=QuantType.QInt8,
    )
    print('INT8 ONNX saved:', int8_path)

## 19. Gợi ý tích hợp backend

Khuyến nghị production:

1. Không chạy model nặng trực tiếp trong request handler chính của Node.js.
2. Tách thành một service riêng, ví dụ Python FastAPI:
   - Node.js gửi text sang `/moderate`
   - FastAPI trả về `ALLOW`, `REVIEW`, `BLOCK`
3. Nếu traffic cao:
   - Dùng queue như BullMQ
   - Cache kết quả moderation cho nội dung trùng lặp
   - Dùng ONNX Runtime / INT8 để giảm latency